---
title: Analyses of feature matrix
date: now
author: Jan Cap
---

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/final/feature_matrix.parquet")
df

## Missing

In [ ]:
# count of missing values per column
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(missing_counts.head(20))

In [ ]:
import plotnine as gg

# Create a binary matrix: 1 = missing, 0 = not missing
missing_matrix = df.isnull().astype(int)

# Correlation between missing patterns (do columns tend to have missing values together?)
missing_corr = missing_matrix.corr()

# Reshape correlation matrix to long format for plotnine
missing_corr_long = missing_corr.reset_index().melt(id_vars="index", var_name="col2", value_name="correlation")
missing_corr_long.columns = ["col1", "col2", "correlation"]

# Get every 10th column name for x-axis breaks
unique_cols = sorted(missing_corr_long["col1"].unique())
x_breaks = [col for i, col in enumerate(unique_cols) if i % 10 == 0]

# Create heatmap of missing value correlations
(
    gg.ggplot(missing_corr_long, gg.aes(x="col2", y="col1", fill="correlation"))
    + gg.geom_tile()
    + gg.scale_fill_gradient2(low="#2166AC", mid="white", high="#B2182B", midpoint=0)
    + gg.scale_x_discrete(breaks=x_breaks)
    + gg.scale_y_discrete(breaks=x_breaks)
    +
    # gg.theme_minimal() +
    gg.theme(
        figure_size=(10, 8),
        axis_text_x=gg.element_text(angle=90, hjust=1, vjust=0.5, size=8),
        plot_title=gg.element_text(hjust=0.5),
        axis_title_x=gg.element_blank(),
        axis_title_y=gg.element_blank(),
    )
    + gg.labs(title="Correlation Between Missing Value Patterns Across Columns", fill="Correlation")
)

From the heatmat of missing value correlations, we can say that for source tables election 2021, election 2025, population and area, when there is missing value for municipality, there is missing whole row (all data for that municipality is missing).


Corelation between 2021 and 2025 election data is about 0.5, which is not very high, but also not very low. Similar applys for population and election 2021. Correlation of missings between population and some area columns is also strong.


In [ ]:
# population vs area correlation
# select only rows that start with "population" or "area" on col1 and col2
pop_area = missing_corr_long[
    ((missing_corr_long["col1"].str.startswith("population")) | (missing_corr_long["col1"].str.startswith("area")))
    & ((missing_corr_long["col2"].str.startswith("population")) | (missing_corr_long["col2"].str.startswith("area")))
]

(
    gg.ggplot(pop_area, gg.aes(x="col2", y="col1", fill="correlation"))
    + gg.geom_tile()
    + gg.scale_fill_gradient2(low="#2166AC", mid="white", high="#B2182B", midpoint=0)
    +
    # gg.theme_minimal() +
    gg.theme(
        figure_size=(10, 8),
        axis_text_x=gg.element_text(angle=90, hjust=1, vjust=0.5, size=8),
        plot_title=gg.element_text(hjust=0.5),
        axis_title_x=gg.element_blank(),
        axis_title_y=gg.element_blank(),
    )
    + gg.labs(title="Correlation Between Missing Value Patterns Across Population and Area Columns", fill="Correlation")
)

Also we can see some correlation between population and area. Especially between all population columns and total area columns. This could indicate that there are some municipalities with missing data. It could be due to statistical offices not reporting data for some municipalities, or it could be due to some municipalities having problems with data collection. 


In [ ]:
# Visualize missing values across observations
# Focus on columns and rows that have missing values
cols_with_missing = missing_counts[missing_counts > 0].index
if len(cols_with_missing) > 0:
    # Get rows that have any missing values
    rows_with_missing = df[cols_with_missing].isnull().any(axis=1)

    # Sample rows for visibility (if too many)
    sample_size = min(100, rows_with_missing.sum())
    missing_rows_idx = df[rows_with_missing].sample(n=sample_size, random_state=42).index

    # Create heatmap of missing values (rows x columns with missing)
    missing_subset = df.loc[missing_rows_idx, cols_with_missing].isnull().astype(int)

    # Reshape for plotnine
    missing_long = missing_subset.reset_index().melt(id_vars="index", var_name="feature", value_name="is_missing")
    missing_long["row_num"] = missing_long.groupby("index").ngroup()
    missing_long["is_missing"] = missing_long["is_missing"].map({0: "Present", 1: "Missing"})

    # Get every 10th feature name for x-axis breaks
    unique_features = sorted(missing_long["feature"].unique())
    x_breaks = [feat for i, feat in enumerate(unique_features) if i % 10 == 0]

    display(
        gg.ggplot(missing_long, gg.aes(x="feature", y="row_num", fill="is_missing"))
        + gg.geom_tile(colour="white", size=0.1)
        + gg.scale_fill_manual(values={"Present": "#66C2A5", "Missing": "#FC8D62"})
        + gg.scale_x_discrete(breaks=x_breaks)
        + gg.theme_minimal()
        + gg.theme(
            figure_size=(14, 8),
            axis_text_x=gg.element_text(angle=90, hjust=1, vjust=0.5, size=8),
            axis_text_y=gg.element_blank(),
            plot_title=gg.element_text(hjust=0.5),
        )
        + gg.labs(
            title=f"Missing Value Patterns: {sample_size} rows with missing values (sample)",
            x="Features with missing values",
            y="Sample of rows",
            fill="Status",
        )
    )

    print("\nMissing value statistics:")
    print(f"Total rows: {len(df)}")
    print(f"Rows with any missing values: {rows_with_missing.sum()} ({100 * rows_with_missing.sum() / len(df):.1f}%)")
    print(f"Columns with missing values: {len(cols_with_missing)}")

## Data correlation

In [ ]:
import plotnine as gg

# Correlation between missing patterns (do columns tend to have missing values together?)
missing_corr = df.corr()

# Reshape correlation matrix to long format for plotnine
missing_corr_long = missing_corr.reset_index().melt(id_vars="index", var_name="col2", value_name="correlation")
missing_corr_long.columns = ["col1", "col2", "correlation"]

# Get every 10th column name for x-axis breaks
unique_cols = sorted(missing_corr_long["col1"].unique())
x_breaks = [col for i, col in enumerate(unique_cols) if i % 10 == 0]

# Create heatmap of missing value correlations
(
    gg.ggplot(missing_corr_long, gg.aes(x="col2", y="col1", fill="correlation"))
    + gg.geom_tile()
    + gg.scale_fill_gradient2(low="#2166AC", mid="white", high="#B2182B", midpoint=0)
    + gg.scale_x_discrete(breaks=x_breaks)
    + gg.scale_y_discrete(breaks=x_breaks)
    +
    # gg.theme_minimal() +
    gg.theme(
        figure_size=(10, 8),
        axis_text_x=gg.element_text(angle=90, hjust=1, vjust=0.5, size=8),
        plot_title=gg.element_text(hjust=0.5),
        axis_title_x=gg.element_blank(),
        axis_title_y=gg.element_blank(),
    )
    + gg.labs(title="Correlation of Value Patterns Across Columns", fill="Correlation")
)

From the correlation graph, migration data are not correlated to any other data (visibly white row and column). There are some correlations between elections 2021 and 2025.
Othervise, there are no strong correlations between any other columns and input datasets.